In [1]:
%pip install -q ultralytics==8.4.1 pycocotools

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import json
import math
import shutil
from collections import defaultdict
from pathlib import Path
from typing import Any
from zipfile import ZIP_STORED, ZipFile

import cv2
import numpy as np
import yaml
from pycocotools import mask as mask_utils
from ultralytics.data.converter import merge_multi_segment

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
BASE = Path("/content/drive/MyDrive/WasteScope")
WORK = Path("/content/wastescope_s2_prepare")
OUTPUT = BASE / "prepared/segmentation_roi"

WORK.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

DETECTOR_REPORT = BASE / "reports/detection/D0_yolo26n_640/val_960"

S1_REPORT = (
    BASE / "reports/segmentation/S1_yolo26n_seg_960" / "val_best_960_conf0.25"
)

Rect = tuple[int, int, int, int]

In [6]:
with ZipFile(BASE / "raw_archives/segmentation_task.zip") as archive:
    annotations = json.loads(
        archive.read("Segmentation Task/labels.json")
    )

with ZipFile(BASE / "prepared/segmentation/train.zip") as archive:
    manifest_bytes = archive.read("video_split.json")

manifest = json.loads(manifest_bytes)

detector_predictions = json.loads(
    (DETECTOR_REPORT / "native_predictions.json").read_text()
)
detector_config = json.loads(
    (DETECTOR_REPORT / "evaluation_config.json").read_text()
)
s1_metrics = json.loads(
    (S1_REPORT / "metrics.json").read_text()
)

In [7]:
overflow = defaultdict(list)
train_boxes = defaultdict(list)

for ann in annotations["annotations"]:
    if ann["category_id"] == 1:
        overflow[ann["image_id"]].append(ann)
    
    else:
        train_boxes[ann["image_id"]].append(ann["bbox"])

records = {
    split: [
        image for image in annotations["images"]
        if image["file_name"].split("_")[0]
        in manifest["splits"][split]
    ] for split in ("train", "val")
}

val_ids = {image["id"] for image in records["val"]}
names_to_ids = {
    image["file_name"]: image["id"]
    for image in records["val"]
}

val_boxes = defaultdict(list)
for prediction in detector_predictions:
    image_id = names_to_ids.get(
        Path(prediction["file_name"]).name
    )
    if image_id is not None and prediction["score"] > 0.05:
        val_boxes[image_id].append(prediction["bbox"])

In [8]:
policy = {
    "experiment": "S2_container_context_v1",
    "class": "Overflow",
    "yolo_class_id": 0,
    "coco_category_id": 1,
    "crop_factor": 3,
    "minimum_side": 512,
    "stored_long_side": 640,
    "crop_edges": "shift_inside_image_then_floor_ceil",
    "duplicate_crops": "remove_exact_bounds",
    "train_proposals": (
        "GT containers; full-frame sample when no container annotation"
    ),
    "val_proposals": (
        "D0 predictions; no full-frame ROI fallback"
    ),
    "detector_run": detector_config["run"],
    "detector_imgsz": 960,
    "detector_confidence": 0.05,
    "detector_checkpoint_sha256": (
        detector_config["checkpoint_sha256"]
    ),
    "full_image_run": s1_metrics["run_name"],
    "full_image_checkpoint_sha256": (
        s1_metrics["checkpoint_sha256"]
    ),
    "fusion": "score-ordered mask-IoU NMS",
    "fusion_iou": 0.5,
    "AP_confidence_min": 0.001,
    "max_detections": 100,
    "supporting_confidence": 0.25,
    "checkpoint_monitor": (
        "canonical_full_image_COCO_mask_mAP50_95"
    ),
    "ultralytics": "8.4.1",
    "crop_labels": (
        "COCO raster masks to external polygons; disconnected "
        "parts joined with merge_multi_segment"
    ),
}

policy["train_jitter"] = {
    "probability": 0.5,
    "translate_fraction": 0.05,
    "scale_range": [0.9, 1.1],
    "seed": 42
}
policy["val_jitter"] = False

print("Training images:", len(records["train"]))
print("Validation images:", len(records["val"]))
print("Output:", OUTPUT)

Training images: 5090
Validation images: 1030
Output: /content/drive/MyDrive/WasteScope/prepared/segmentation_roi


In [9]:
def context_crop(bbox, width, height, *, rng, jitter) -> Rect:
    x, y, w, h = bbox
    cx, cy = x + w / 2, y + h / 2
    side = max(3 * w, 3 * h, 512)
    
    if (rng is not None and jitter is not None and rng.random() < jitter["probability"]):
        side = max(
            512, side * rng.uniform(*jitter["scale_range"]),
        )
        
        shift = jitter["translate_fraction"]
        cx += rng.uniform(-shift, shift) * min(side, width)
        cy += rng.uniform(-shift, shift) * min(side, height)
        
    crop_width = min(side, width)
    crop_height = min(side, height)
    
    left = min(
        max(cx - crop_width / 2, 0), width - crop_width
    )
    top = min(
        max(cy - crop_height / 2, 0), height - crop_height
    )
    
    return (
        math.floor(left), math.floor(top),
        math.ceil(left + crop_width), math.ceil(top + crop_height)
    )
    


def crop_labels(masks, rect):
    x1, y1, x2, y2 = rect
    rows = []
    joined = 0
    tiny = 0
    
    for mask in masks:
        clipped = np.ascontiguousarray(mask[y1:y2, x1:x2])
        
        contours, _ = cv2.findContours(
            clipped,
            cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        segments = [
            contour.reshape(-1, 2)
            for contour in contours
            if len(contour) >= 3 and cv2.contourArea(contour) > 0
        ]
        
        if not segments:
            tiny += int(bool(clipped.any()))
            continue

        if len(segments) > 1:
            points = np.concatenate(
                merge_multi_segment([
                    segment.reshape(-1).tolist()
                    for segment in segments
                ])
            )
            joined += 1
        else:
            points = segments[0]

        points = points.astype(float) / np.array([x2-x1, y2-y1])
        rows.append(
            "0 " + " ".join(
                f"{value:.8f}" for value in points.reshape(-1)
            )
        )

    text = "\n".join(rows) + ("\n" if rows else "")
    return text, joined, tiny

In [11]:
def build_split(split, records, boxes_by_image, overflow_by_image, source_zip, output_zip, policy):
    index = {}
    counts = {
        "images": len(records),
        "crops": 0,
        "positive_crops": 0,
        "full_frame_fallbacks": 0,
        "duplicate_crops_removed": 0,
        "joined_instances": 0,
        "tiny_visible_instances_omitted": 0
    }
    jitter = policy["train_jitter"] if split == "train" else None
    rng = np.random.default_rng(jitter["seed"]) if jitter is not None else None
    
    with (ZipFile(source_zip) as source,
        ZipFile(output_zip, "w", ZIP_STORED) as target
    ):
        for number, image in enumerate(records, 1):
            image_id = image["id"]
            width = image["width"]
            height = image["height"]

            rectangles = [
                context_crop(
                    box, width, height, rng=rng, jitter=jitter,
                )
                for box in boxes_by_image.get(image_id, [])
            ]
            unique = list(dict.fromkeys(rectangles))
            counts["duplicate_crops_removed"] += (len(rectangles) - len(unique))

            if split == "train" and not unique:
                unique = [(0, 0, width, height)]
                counts["full_frame_fallbacks"] += 1

            if unique:
                payload = source.read(f"{split}/images/{image['file_name']}")
                frame = cv2.imdecode(
                    np.frombuffer(payload, dtype=np.uint8), cv2.IMREAD_COLOR
                )

                masks = [
                    mask_utils.decode(
                        mask_utils.merge(mask_utils.frPyObjects(ann["segmentation"], height, width))
                    )
                    for ann in overflow_by_image.get(image_id, [])
                ]

                for crop_number, rect in enumerate(unique):
                    x1, y1, x2, y2 = rect
                    crop = frame[y1:y2, x1:x2]
                    scale = 640 / max(crop.shape[:2])
                    
                    size = (round(crop.shape[1] * scale), round(crop.shape[0] * scale))
                    crop = cv2.resize(crop, size, interpolation=cv2.INTER_LINEAR)
                    text, joined, tiny = crop_labels(masks, rect)
                    
                    stem = (
                        f"{Path(image['file_name']).stem}"
                        f"__roi{crop_number:03d}"
                    )
                    
                    success, encoded = cv2.imencode(".png", crop, [cv2.IMWRITE_PNG_COMPRESSION, 1])
                    if not success:
                        raise RuntimeError(f"Could not encode {stem}")

                    target.writestr(f"{split}/images/{stem}.png", encoded.tobytes())
                    target.writestr(f"{split}/labels/{stem}.txt", text)
                    
                    index[stem] = {
                        "image_id": image_id,
                        "box": list(rect),
                        "image_shape": [height, width],
                    }

                    counts["crops"] += 1
                    counts["positive_crops"] += int(bool(text))
                    counts["joined_instances"] += joined
                    counts["tiny_visible_instances_omitted"] += tiny

            if number % 250 == 0 or number == len(records):
                print(
                    f"{split}: {number}/{len(records)} images; "
                    f"{counts['crops']} crops", flush=True
                )

        target.writestr(f"{split}/roi_index.json", json.dumps(index))
        target.writestr("roi_policy.json", json.dumps(policy, indent=2))

    counts["negative_crops"] = counts["crops"] - counts["positive_crops"]
    return counts

In [12]:
summary = {"policy": policy,"splits": {}}

for split in ("train", "val"):
    local_source = WORK / f"source_{split}.zip"
    local_output = WORK / f"{split}.zip"

    print(f"Copying source {split}.zip locally", flush=True)
    if not local_source.is_file():
        shutil.copyfile(BASE / f"prepared/segmentation/{split}.zip", local_source)

    summary["splits"][split] = build_split(
        split=split,
        records=records[split],
        boxes_by_image=(train_boxes if split == "train" else val_boxes),
        overflow_by_image=overflow,
        source_zip=local_source,
        output_zip=local_output,
        policy=policy
    )

    with ZipFile(local_output, "a", ZIP_STORED) as target:
        target.writestr(
            f"{split}/preparation_summary.json", json.dumps(summary["splits"][split], indent=2),
        )

        if split == "train":
            target.writestr("video_split.json", manifest_bytes)
            target.writestr(
                "data.yaml",
                yaml.safe_dump({
                    "train": "train/images",
                    "val": "val/images",
                    "names": {0: "Overflow"},
                }, sort_keys=False)
            )
            target.write(S1_REPORT / "predictions.json", "support/s1_val_predictions.json")
            target.write(S1_REPORT / "metrics.json","support/s1_val_metrics.json")
        else:
            original_gt = {
                "info": annotations.get("info", {}),
                "licenses": annotations.get("licenses", []),
                "images": records["val"],
                "categories": [
                    category for category in annotations["categories"] if category["id"] == 1
                ],
                "annotations": [
                    ann for ann in annotations["annotations"]
                    if ann["image_id"] in val_ids and ann["category_id"] == 1
                ]
            }
            target.writestr("val/annotations.json", json.dumps(original_gt),)

    pending = OUTPUT / f"{split}.zip.part"
    print(f"Saving {split}.zip to Drive", flush=True)
    shutil.copyfile(local_output, pending)
    pending.replace(OUTPUT / f"{split}.zip")

    local_source.unlink()
    local_output.unlink()
    print(f"Saved {split}.zip", flush=True)

(OUTPUT / "roi_policy.json").write_text(json.dumps(policy, indent=2), encoding="utf-8")
(OUTPUT / "preparation_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary["splits"], indent=2))

Copying source train.zip locally
train: 250/5090 images; 432 crops
train: 500/5090 images; 822 crops
train: 750/5090 images; 2518 crops
train: 1000/5090 images; 3940 crops
train: 1250/5090 images; 5657 crops
train: 1500/5090 images; 6779 crops
train: 1750/5090 images; 8004 crops
train: 2000/5090 images; 9325 crops
train: 2250/5090 images; 10803 crops
train: 2500/5090 images; 11890 crops
train: 2750/5090 images; 12473 crops
train: 3000/5090 images; 13028 crops
train: 3250/5090 images; 13998 crops
train: 3500/5090 images; 15372 crops


TypeError: 'NoneType' object is not subscriptable